# Structured JSON â†’ Markdown-per-Page

Nimmt die strukturierten JSONs aus `structured_export.ipynb` und vereinfacht sie zu **einem Markdown-String pro Seite**.

| Vorher | Nachher |
|---|---|
| `content[].content[]` â€” Block-Array mit Typen, IDs, Referenzen | `content.pages[]` â€” ein `text`-String (Markdown) pro Seite |
| `tables[]` â€” separates Array mit 3 Formaten | Tabellen-Markdown inline im Seitentext |

## Formatting-Regeln

| Block-Typ | Markdown |
|---|---|
| `page_header` | `# text` (erste 3 BlÃ¶cke) / `## text` |
| `section_header` | `# text` (erster Content-Block) / `## text` |
| `paragraph` | `### text` |
| `table` | Markdown-Tabelle aus tables-Array |
| `list_item` | `- text` |
| `text`, `caption`, `footnote` | plain text |

## Gruppierung
- **Tabellen-Gruppe**: Text mit `:` am Ende â†’ Tabelle â†’ FuÃŸnoten
- **Listen-Gruppe**: Text mit `:` am Ende â†’ aufeinanderfolgende list_items

## Gefiltert
- `page_footer`, `picture`, Separator-Zeilen (`___...`)

In [1]:
import json
import re
import datetime
from pathlib import Path
from typing import List, Dict, Optional, Tuple
from tqdm import tqdm

In [2]:
GLYPH_PATTERNS = [
    (re.compile(r'/zero\.tnum'),  '0'),
    (re.compile(r'/one\.tnum'),   '1'),
    (re.compile(r'/two\.tnum'),   '2'),
    (re.compile(r'/three\.tnum'), '3'),
    (re.compile(r'/four\.tnum'),  '4'),
    (re.compile(r'/five\.tnum'),  '5'),
    (re.compile(r'/six\.tnum'),   '6'),
    (re.compile(r'/seven\.tnum'), '7'),
    (re.compile(r'/eight\.tnum'), '8'),
    (re.compile(r'/nine\.tnum'),  '9'),
    (re.compile(r'/dollar\.pl'),  '$'),
    (re.compile(r'/percent'),      '%'),
    (re.compile(r'/([A-Z])\.cap'), lambda m: m.group(1)),
    (re.compile(r'/([a-z])\.lc'),  lambda m: m.group(1)),
    (re.compile(r'glyph<[^>]*>'),  ''),
    (re.compile(r'/[a-z]+\.(lf|sc|smcp|onum|pnum|subs|sups|tnum|pl)'), ''),
    (re.compile(r' *\.{2,}'), ''),   # dot-leader fill characters
    (re.compile('\ufffd'), ''),       # unicode replacement character
]

FILTERED_TYPES = frozenset({"page_footer", "picture"})
KNOWN_TYPES = frozenset({
    "page_header", "section_header", "paragraph", "table", "list_item",
    "text", "caption", "footnote", "checkbox_selected", "checkbox_unselected",
    "page_footer", "picture",
})
SEPARATOR_RE = re.compile(r'^[_\-]{5,}$')


class ReportMerger:
    """Structured JSON -> Markdown-per-page JSON."""

    def __init__(self, source_dir: str, output_dir: str = None):
        self.source_dir = Path(source_dir)
        if output_dir:
            self.output_dir = Path(output_dir)
        else:
            self.output_dir = self.source_dir.parent / (self.source_dir.name + "_merged")
        self.output_dir.mkdir(parents=True, exist_ok=True)
        print(f"Source: {self.source_dir}")
        print(f"Output: {self.output_dir}")

    # -- Text cleanup -----------------------------------------------------

    @staticmethod
    def _clean_text(text: str, corrections: List) -> str:
        for pattern, repl in GLYPH_PATTERNS:
            for match in pattern.finditer(text):
                corrections.append(match.group())
            text = pattern.sub(repl, text)
        return text

    @staticmethod
    def _is_separator(text: str) -> bool:
        return bool(SEPARATOR_RE.match(text.strip()))

    # -- Lookahead helpers ------------------------------------------------

    @staticmethod
    def _next_content_block(blocks: List[Dict], start: int) -> Optional[Dict]:
        """Return next block that isn't filtered or a separator."""
        for j in range(start, len(blocks)):
            b = blocks[j]
            btype = b.get("type")
            if btype in FILTERED_TYPES:
                continue
            if btype == "paragraph" and ReportMerger._is_separator(b.get("text", "")):
                continue
            return b
        return None

    @staticmethod
    def _block_text_ends_colon(block: Dict) -> bool:
        text = block.get("text", "")
        return text.rstrip().endswith(":")

    # -- Group detection --------------------------------------------------

    def _is_table_group_start(self, block: Dict, blocks: List, i: int) -> bool:
        if block.get("type") in FILTERED_TYPES or block.get("type") == "table":
            return False
        if not self._block_text_ends_colon(block):
            return False
        nxt = self._next_content_block(blocks, i + 1)
        return nxt is not None and nxt.get("type") == "table"

    def _is_list_group_start(self, block: Dict, blocks: List, i: int) -> bool:
        if block.get("type") in FILTERED_TYPES or block.get("type") in ("table", "list_item"):
            return False
        if not self._block_text_ends_colon(block):
            return False
        nxt = self._next_content_block(blocks, i + 1)
        return nxt is not None and nxt.get("type") == "list_item"

    # -- Group rendering --------------------------------------------------

    def _render_table_group(self, block: Dict, blocks: List, i: int,
                            table_map: Dict, sha1: str) -> Tuple[List[str], int]:
        """Render intro + table + footnotes. Returns (lines, next_index)."""
        lines = [f"### {block.get('text', '')}"]
        j = i + 1
        # skip to table
        while j < len(blocks):
            b = blocks[j]
            btype = b.get("type")
            if btype in FILTERED_TYPES or (btype == "paragraph" and self._is_separator(b.get("text", ""))):
                j += 1
                continue
            if btype == "table":
                tid = b.get("table_id")
                if tid not in table_map:
                    raise ValueError(f"Table with ID={tid} not found in report {sha1}!")
                lines.append(table_map[tid])
                j += 1
                break
            break  # unexpected block, stop
        # collect trailing separators + footnotes
        while j < len(blocks):
            b = blocks[j]
            btype = b.get("type")
            if btype in FILTERED_TYPES:
                j += 1
                continue
            if btype == "paragraph" and self._is_separator(b.get("text", "")):
                j += 1
                continue
            if btype == "footnote":
                lines.append(b.get("text", ""))
                j += 1
                continue
            break
        return lines, j

    def _render_list_group(self, block: Dict, blocks: List, i: int) -> Tuple[List[str], int]:
        """Render intro + consecutive list_items. Returns (lines, next_index)."""
        lines = [f"### {block.get('text', '')}"]
        j = i + 1
        while j < len(blocks):
            b = blocks[j]
            btype = b.get("type")
            if btype in FILTERED_TYPES:
                j += 1
                continue
            if btype == "list_item":
                lines.append(f"- {b.get('text', '')}")
                j += 1
                continue
            break
        return lines, j

    # -- Block rendering --------------------------------------------------

    def _is_first_content_block(self, blocks: List[Dict], index: int) -> bool:
        """True if index is the first non-filtered, non-page_header block."""
        for j in range(len(blocks)):
            b = blocks[j]
            if b.get("type") in FILTERED_TYPES or b.get("type") == "page_header":
                continue
            return j == index
        return False

    def _render_block(self, block: Dict, index: int, has_page_header: bool,
                      blocks: List[Dict], table_map: Dict, sha1: str) -> Optional[str]:
        btype = block.get("type")
        text = block.get("text", "")

        if btype in FILTERED_TYPES:
            return None

        if btype not in KNOWN_TYPES:
            raise ValueError(f"Unknown block type: {btype}")

        if btype == "page_header":
            return f"# {text}" if index < 3 else f"## {text}"

        if btype == "section_header":
            if not has_page_header and self._is_first_content_block(blocks, index):
                return f"# {text}"
            return f"## {text}"

        if btype == "paragraph":
            if self._is_separator(text):
                return None
            return f"### {text}"

        if btype == "table":
            tid = block.get("table_id")
            if tid not in table_map:
                raise ValueError(f"Table with ID={tid} not found in report {sha1}!")
            return table_map[tid]

        if btype == "list_item":
            return f"- {text}"

        if btype == "checkbox_selected":
            return f"- [x] {text}"

        if btype == "checkbox_unselected":
            return f"- [ ] {text}"

        # text, caption, footnote
        return text

    # -- Page processing --------------------------------------------------

    def _process_page(self, page: Dict, table_map: Dict, sha1: str,
                      corrections: List) -> str:
        blocks = page.get("content", [])
        if not blocks:
            return ""

        has_page_header = blocks[0].get("type") == "page_header"
        lines = []
        i = 0

        while i < len(blocks):
            block = blocks[i]
            btype = block.get("type")

            if btype in FILTERED_TYPES:
                i += 1
                continue

            if btype == "paragraph" and self._is_separator(block.get("text", "")):
                i += 1
                continue

            # Table group?
            if self._is_table_group_start(block, blocks, i):
                group_lines, i = self._render_table_group(block, blocks, i, table_map, sha1)
                lines.extend(group_lines)
                continue

            # List group?
            if self._is_list_group_start(block, blocks, i):
                group_lines, i = self._render_list_group(block, blocks, i)
                lines.extend(group_lines)
                continue

            rendered = self._render_block(block, i, has_page_header, blocks, table_map, sha1)
            if rendered is not None:
                lines.append(rendered)
            i += 1

        text = "\n\n".join(lines)
        return self._clean_text(text, corrections).strip()

    # -- Document processing ----------------------------------------------

    def _process_document(self, data: Dict) -> Dict:
        table_map = {t["table_id"]: t["markdown"] for t in data.get("tables", [])}
        sha1 = data["metainfo"].get("sha1_name", "unknown")
        corrections = []

        pages = []
        for page in data.get("content", []):
            text = self._process_page(page, table_map, sha1, corrections)
            pages.append({"page": page["page"], "text": text})

        if corrections:
            print(f"  Fixed {len(corrections)} glyph occurrences in {sha1}")
            print(f"  {corrections[:30]}")

        return {
            "metainfo": data["metainfo"],
            "content": {
                "chunks": None,
                "pages": pages,
            },
        }

    # -- Batch processing -------------------------------------------------

    def process_all(self) -> Dict:
        json_files = sorted(self.source_dir.glob("*.json"))
        print(f"{len(json_files)} JSON files\n")

        stats = {"success": 0, "error": 0}

        for jf in tqdm(json_files, desc="Merging"):
            try:
                with open(jf, "r", encoding="utf-8") as f:
                    data = json.load(f)

                result = self._process_document(data)

                out_file = self.output_dir / jf.name
                with open(out_file, "w", encoding="utf-8") as f:
                    json.dump(result, f, indent=2, ensure_ascii=False)

                pages = result["content"]["pages"]
                non_empty = sum(1 for p in pages if p["text"])
                tqdm.write(f"  {jf.stem}: {len(pages)} pages ({non_empty} non-empty)")
                stats["success"] += 1

            except Exception as e:
                tqdm.write(f"  {jf.stem}: Error — {e}")
                stats["error"] += 1

        print(f"\nOK: {stats['success']} | Errors: {stats['error']}")
        return stats

In [3]:
merger = ReportMerger(
    source_dir="C:/Users/phili/PycharmProjects/doc/data/src/docling_output/20260409_195022",
)
merger.process_all()

Source: C:\Users\phili\PycharmProjects\doc\data\src\docling_output\20260409_195022
Output: C:\Users\phili\PycharmProjects\doc\data\src\docling_output\20260409_195022_merged
50 JSON files



Merging:   2%|▏         | 1/50 [00:00<00:06,  7.51it/s]

  Fixed 3 glyph occurrences in 148bc4c17f5aa13b04fee25ee8e1da9269708034
  [' ................................................................................................................................................', ' ..................................................................................................................................', ' ......................................................................................................................................']
  CE_2012: 159 pages (159 non-empty)


Merging:   4%|▍         | 2/50 [00:00<00:07,  6.51it/s]

  CE_2013: 172 pages (171 non-empty)


...

Merging:  96%|█████████▌| 48/50 [00:05<00:00, 12.08it/s]

  DRE_2016: 146 pages (146 non-empty)
  Fixed 3 glyph occurrences in 5e077cb97ecfb677fe6e42b77357bf0d460a7c0e
  [' ...', '...', '...']
  DVN_2004: 112 pages (111 non-empty)


Merging: 100%|██████████| 50/50 [00:05<00:00,  8.83it/s]

  DVN_2007: 116 pages (115 non-empty)
  Fixed 15 glyph occurrences in f3f8d202690078f841f7abc0689c83292c9deeb8
  [' ...........................................', ' ..................', ' .............................................', ' ..............................', ' .................................', ' ...................................', ' .........................', ' .............................', ' ...........', ' ..............................', ' .................................', ' ...................................', ' .........................', ' .............................', ' ...........']
  DVN_2010: 154 pages (154 non-empty)

OK: 48 | Errors: 2


{'success': 48, 'error': 2}